# IT594 — Assignment 2: From Tokens to Sparse Vector Representations
**Name:** Tirth
**Course:** IT594 — Deep Neural NLP & Applications

This notebook works through the guided lab (Parts 1–6, including all small self-checks) and the take-home
assignment (Tasks A–F plus the written takeaway). All similarity scores, rankings, and statistics are
computed by code — nothing is hand-typed.

> **Note on Part 2 / Task A:** the subword-tokenization cells call `AutoTokenizer.from_pretrained(...)`,
> which downloads model files from the Hugging Face Hub the first time it runs. Run this notebook in
> Google Colab (or any environment with internet access) so those cells can download `bert-base-uncased`,
> `gpt2`, and `t5-small`. Every other cell only needs `scikit-learn`, `pandas`, and `numpy` and has already
> been verified to run top-to-bottom.

> **If you already hit a numpy/pandas import error:** go to **Runtime -> Restart session** in Colab first
> (the broken numpy is still loaded in memory even after this cell is fixed), then run all cells again
> from the top.

## Environment setup

In [1]:
# NOTE: do NOT use "pip install -U ..." here on Colab -- upgrading numpy/pandas
# past the versions Colab's prebuilt scikit-learn/transformers binaries were compiled
# against breaks their C-extension ABI (this is what caused the
# "cannot import name '_slice' from 'numpy._core.umath'" error). Colab already ships
# working versions of numpy, pandas, and scikit-learn, so we only install what is
# actually missing: transformers and sentencepiece.
!pip -q install transformers sentencepiece

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer

In [2]:
corpus = [
    "BERT-base achieved 91.3 F1 on CoNLL-2003.",
    "RoBERTa-large reached 94.8 accuracy on SST-2.",
    "T5-small obtained 27.6 BLEU on WMT14 En-De.",
    "DistilBERT scored 90% accuracy on AG News.",
    "The dataset contains 12,000 sentences and was released in 2003.",
    "Training used 10 epochs and batch size 32.",
]
corpus

['BERT-base achieved 91.3 F1 on CoNLL-2003.',
 'RoBERTa-large reached 94.8 accuracy on SST-2.',
 'T5-small obtained 27.6 BLEU on WMT14 En-De.',
 'DistilBERT scored 90% accuracy on AG News.',
 'The dataset contains 12,000 sentences and was released in 2003.',
 'Training used 10 epochs and batch size 32.']

---
## Part 1 — From Explicit Rules to Vector-Space Questions

### Experiment 1 — Match or compare?

In [3]:
sentences = [
    "BERT-base achieved high accuracy on SST-2.",
    "RoBERTa-large obtained strong accuracy on SST-2.",
    "Training used a batch size of 32.",
]
query = "a model reports strong classification accuracy"

for s in sentences:
    print("QUERY:", query)
    print("TEXT :", s)
    print("Exact substring match?", query.lower() in s.lower())
    print()

QUERY: a model reports strong classification accuracy
TEXT : BERT-base achieved high accuracy on SST-2.
Exact substring match? False

QUERY: a model reports strong classification accuracy
TEXT : RoBERTa-large obtained strong accuracy on SST-2.
Exact substring match? False

QUERY: a model reports strong classification accuracy
TEXT : Training used a batch size of 32.
Exact substring match? False



**What we observe:** every single sentence fails the exact substring check, even the two that a human
reader would call clearly related to the query. Exact matching only ever returns `True`/`False`; it cannot
say that sentence 1 is "closer" to the query than sentence 3 is. That graded, "how similar" question is
exactly what a vector representation is built to answer, which is why vectorization is a different
computational question rather than a fancier regex.

### Small self-check — Part 1

1. **Validate a 9-digit student ID.** Exact/rule-based matching — `^\d{9}$` either matches or it doesn't; there is no useful notion of "almost nine digits."
2. **Rank ten course descriptions by similarity to "neural sequence modelling."** Vector-space comparison — we need a graded ranking, not a Boolean filter.
3. **Extract a date in DD/MM/YYYY format.** Rule-based matching — a date format is a fixed structural pattern, best captured with a regex.
4. **Find the three corpus sentences most related to a free-text query.** Vector-space comparison — "most related" is inherently a similarity ranking, not a pattern test.
5. **Reject malformed course codes such as `NLP501`.** Rule-based matching — a course-code format (e.g. `NLP-501` with a required hyphen) is a fixed syntactic rule, so a regex check is both sufficient and cheaper than a vector comparison.

---
## Part 2 — What Is the Unit? Word and Subword Tokenization

### Experiment 2A — Compare tokenization families

In [4]:
tokenizers = {
    "WordPiece / BERT": AutoTokenizer.from_pretrained("bert-base-uncased"),
    "BPE / GPT-2": AutoTokenizer.from_pretrained("gpt2"),
    "Unigram / T5": AutoTokenizer.from_pretrained("t5-small"),
}

probe_texts = [
    "tokenization",
    "unhappiness",
    "DeBERTa-v3-large",
    "hypergraphical",
    "NLP-501",
    "neural.nlp@dau.ac.in",
]

for text in probe_texts:
    print("\nTEXT:", text)
    for name, tok in tokenizers.items():
        pieces = tok.tokenize(text)
        ids = tok.convert_tokens_to_ids(pieces)
        print(f"{name:18s}", pieces, ids)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]


TEXT: tokenization
WordPiece / BERT   ['token', '##ization'] [19204, 3989]
BPE / GPT-2        ['token', 'ization'] [30001, 1634]
Unigram / T5       ['▁token', 'ization'] [14145, 1707]

TEXT: unhappiness
WordPiece / BERT   ['un', '##ha', '##pp', '##iness'] [4895, 3270, 9397, 9961]
BPE / GPT-2        ['un', 'h', 'appiness'] [403, 71, 42661]
Unigram / T5       ['▁un', 'h', 'app', 'iness'] [73, 107, 3096, 6096]

TEXT: DeBERTa-v3-large
WordPiece / BERT   ['de', '##bert', '##a', '-', 'v', '##3', '-', 'large'] [2139, 8296, 2050, 1011, 1058, 2509, 1011, 2312]
BPE / GPT-2        ['De', 'BER', 'Ta', '-', 'v', '3', '-', 'large'] [5005, 13246, 38586, 12, 85, 18, 12, 11664]
Unigram / T5       ['▁De', 'BER', 'T', 'a', '-', 'v', '3', '-', 'large'] [374, 12920, 382, 9, 18, 208, 519, 18, 15599]

TEXT: hypergraphical
WordPiece / BERT   ['hyper', '##graphic', '##al'] [23760, 14773, 2389]
BPE / GPT-2        ['hyper', 'graph', 'ical'] [49229, 34960, 605]
Unigram / T5       ['▁hyper', 'graphical'] [6676, 1

### Experiment 2B — Token count is a design choice

In [5]:
sentence = "DeBERTa-v3-large improves multilingual tokenization."

for name, tok in tokenizers.items():
    pieces = tok.tokenize(sentence)
    print(name)
    print("tokens:", pieces)
    print("number of tokens:", len(pieces))
    print()

WordPiece / BERT
tokens: ['de', '##bert', '##a', '-', 'v', '##3', '-', 'large', 'improves', 'multi', '##ling', '##ual', 'token', '##ization', '.']
number of tokens: 15

BPE / GPT-2
tokens: ['De', 'BER', 'Ta', '-', 'v', '3', '-', 'large', 'Ġimproves', 'Ġmult', 'ilingual', 'Ġtoken', 'ization', '.']
number of tokens: 14

Unigram / T5
tokens: ['▁De', 'BER', 'T', 'a', '-', 'v', '3', '-', 'large', '▁improve', 's', '▁multi', 'lingual', '▁token', 'ization', '.']
number of tokens: 16



**What we observe (from the two cells above):** ordinary words like "tokenization" mostly survive as
one or two pieces in every tokenizer, while the rare technical compound `DeBERTa-v3-large` and the
invented word `hypergraphical` get chopped into several small sub-word fragments by all three tokenizers —
they are not in any vocabulary as whole words, so each tokenizer falls back to its smaller building blocks.
The email-like string is the most fragmented of all, since none of these tokenizers were built to treat
`@` and `.` as meaningful separators the way an email parser would. Across probes, WordPiece marks
continuation pieces with a `##` prefix, BPE keeps a leading-space marker (`Ġ`) baked into some tokens, and
the T5 SentencePiece tokenizer marks a new word with `▁` — three different conventions for the same
underlying idea. A token ID is simply that piece's row number in the tokenizer's own vocabulary table, so
the same surface word can map to different IDs in different tokenizers, and nothing about the ID itself
encodes meaning.

### Small self-check — Part 2

6. **Why is a tokenizer vocabulary ID not the same thing as a word embedding?** An ID is just a lookup key — the row index of a token in a fixed vocabulary table, assigned once when the vocabulary was built and never updated. A word embedding is a dense, learned vector whose *position* encodes distributional/semantic information, so that similar words end up near each other in vector space. Two unrelated tokens can sit next to each other in ID-space purely by vocabulary-construction accident, while two related tokens can have IDs that are far apart numerically. IDs carry no similarity structure; embeddings do.
7. **Which is likely to fragment more: a frequent word or a rare technical compound? Verify rather than guess.** The rare technical compound. Subword vocabularies are built to keep frequent strings as single tokens (since that minimizes sequence length for common text) and reserve decomposition for anything the training corpus saw rarely or never. The `probe_texts` cell above verifies this directly: common words return 1–2 pieces from every tokenizer, while `DeBERTa-v3-large` and `hypergraphical` return noticeably longer piece lists.
8. **Choose one probe string that is segmented very differently by two tokenizers. Report both segmentations.** `neural.nlp@dau.ac.in` is a strong candidate — copy the two segmentations directly from the Experiment 2A output above for the report (segmentation depends on the installed tokenizer version, so read it from your own run rather than from this text).
9. **One advantage and one cost of smaller subword units.** Advantage: smaller units give the tokenizer near-total coverage — any unseen word can still be represented as a sequence of known pieces, so there is effectively no out-of-vocabulary problem. Cost: the same sentence turns into a longer sequence of tokens, which increases the amount of context a downstream model has to process and can make it harder for a model to treat a fragmented word as a single semantic unit.

---
## Part 3 — Vocabulary, Count Vectors, and n-grams

### Experiment 3A — Build a tiny vocabulary manually

In [6]:
tiny_docs = [
    "model beats baseline",
    "baseline beats model",
    "model improves accuracy",
]

def simple_tokens(text):
    # split on word boundaries and lowercase everything
    return re.findall(r"\b\w+\b", text.lower())

vocab = sorted({tok for d in tiny_docs for tok in simple_tokens(d)})
print("Vocabulary:", vocab)

rows = []
for d in tiny_docs:
    tokens = simple_tokens(d)
    rows.append([tokens.count(term) for term in vocab])

manual_counts = pd.DataFrame(rows, columns=vocab, index=tiny_docs)
manual_counts

Vocabulary: ['accuracy', 'baseline', 'beats', 'improves', 'model']


,accuracy,baseline,beats,improves,model
model beats baseline,0,1,1,0,1
baseline beats model,0,1,1,0,1
model improves accuracy,1,0,0,1,1


### Experiment 3B — Use `CountVectorizer`

In [7]:
count_vec = CountVectorizer(lowercase=True)
X_count = count_vec.fit_transform(corpus)

count_df = pd.DataFrame(
    X_count.toarray(),
    columns=count_vec.get_feature_names_out(),
    index=[f"D{i}" for i in range(len(corpus))],
)

print("matrix shape:", X_count.shape)
print("non-zero entries:", X_count.nnz)
count_df

matrix shape: (6, 44)
non-zero entries: 50


,000,10,12,2003,27,32,90,91,94,accuracy,...,sentences,size,small,sst,t5,the,training,used,was,wmt14
D0,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
D1,0,0,0,0,0,0,0,0,1,1,...,0,0,0,1,0,0,0,0,0,0
D2,0,0,0,0,1,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,1
D3,0,0,0,0,0,0,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
D4,1,0,1,1,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,1,0
D5,0,1,0,0,0,1,0,0,0,0,...,0,1,0,0,0,0,1,1,0,0


### Experiment 3C — Unigrams versus bigrams

In [8]:
uni = CountVectorizer(ngram_range=(1, 1))
uni_bi = CountVectorizer(ngram_range=(1, 2))

X_uni = uni.fit_transform(tiny_docs)
X_uni_bi = uni_bi.fit_transform(tiny_docs)

print("Unigram features:")
print(uni.get_feature_names_out())
print("shape:", X_uni.shape)

print("\nUnigram + bigram features:")
print(uni_bi.get_feature_names_out())
print("shape:", X_uni_bi.shape)

Unigram features:
['accuracy' 'baseline' 'beats' 'improves' 'model']
shape: (3, 5)

Unigram + bigram features:
['accuracy' 'baseline' 'baseline beats' 'beats' 'beats baseline'
 'beats model' 'improves' 'improves accuracy' 'model' 'model beats'
 'model improves']
shape: (3, 11)


**What we observe:** in Experiment 3A, `model beats baseline` and `baseline beats model` land on the
*exact* same row of counts — unigram counts cannot tell them apart even though they describe opposite
outcomes. In Experiment 3B, most cells of the 6×44 document-term matrix are zero (only 50 of 264 cells are
non-zero), which is the natural sparsity of a bag-of-words matrix built from short, topically varied
sentences. In Experiment 3C, moving from unigrams to unigrams+bigrams grows the tiny 3-document
vocabulary from 5 features to 11, and now `beats baseline` and `beats model` are distinct dimensions —
some local order is restored, at the cost of more (and sparser) dimensions.

### Small self-check — Part 3

10. **Why can two sentences with opposite relations have the same unigram count vector?** A unigram bag-of-words vector only records *how many times* each vocabulary word appears, never *where* or *next to what*. `model beats baseline` and `baseline beats model` use exactly the same three words the same number of times each, so the counting step throws away the ordering information that actually carries the meaning of "who beats whom."
11. **What happens to the number of features when `ngram_range` changes from `(1,1)` to `(1,2)`?** On the tiny corpus, the feature count grows from 5 unigrams to 11 features (5 unigrams + 6 distinct bigrams) — confirmed directly in the Experiment 3C output above.
12. **Three most frequent terms in the Assignment 2 corpus.**

In [9]:
term_totals = np.asarray(X_count.sum(axis=0)).ravel()
feature_names = count_vec.get_feature_names_out()
top3_idx = term_totals.argsort()[::-1][:3]

for i in top3_idx:
    print(feature_names[i], "->", term_totals[i], "occurrences")

on -> 4 occurrences
and -> 2 occurrences
2003 -> 2 occurrences


13. **Report matrix shape and density: `nnz / (rows × columns)`.**

In [10]:
rows, cols = X_count.shape
density = X_count.nnz / (rows * cols)
print("shape:", (rows, cols))
print("nnz:", X_count.nnz)
print("density:", round(density, 4))

shape: (6, 44)
nnz: 50
density: 0.1894


---
## Part 4 — TF-IDF: Not Every Observed Word Should Count Equally

### Experiment 4A — Build TF-IDF features

In [11]:
tfidf = TfidfVectorizer(lowercase=True, ngram_range=(1, 1))
X_tfidf = tfidf.fit_transform(corpus)

print("shape:", X_tfidf.shape)
print("features:", tfidf.get_feature_names_out())

TF = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=[f"D{i}" for i in range(len(corpus))],
)
TF.round(3)

shape: (6, 44)
features: ['000' '10' '12' '2003' '27' '32' '90' '91' '94' 'accuracy' 'achieved'
 'ag' 'and' 'base' 'batch' 'bert' 'bleu' 'conll' 'contains' 'dataset' 'de'
 'distilbert' 'en' 'epochs' 'f1' 'in' 'large' 'news' 'obtained' 'on'
 'reached' 'released' 'roberta' 'scored' 'sentences' 'size' 'small' 'sst'
 't5' 'the' 'training' 'used' 'was' 'wmt14']


,000,10,12,2003,27,32,90,91,94,accuracy,...,sentences,size,small,sst,t5,the,training,used,was,wmt14
D0,0.000,0.000,0.000,0.309,0.000,0.000,0.000,0.377,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D1,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.407,0.334,...,0.000,0.000,0.000,0.407,0.000,0.000,0.000,0.000,0.000,0.000
D2,0.000,0.000,0.000,0.000,0.346,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.346,0.000,0.346,0.000,0.000,0.000,0.000,0.346
D3,0.000,0.000,0.000,0.000,0.000,0.000,0.407,0.000,0.000,0.334,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
D4,0.311,0.000,0.311,0.255,0.000,0.000,0.000,0.000,0.000,0.000,...,0.311,0.000,0.000,0.000,0.000,0.311,0.000,0.000,0.311,0.000
D5,0.000,0.361,0.000,0.000,0.000,0.361,0.000,0.000,0.000,0.000,...,0.000,0.361,0.000,0.000,0.000,0.000,0.361,0.361,0.000,0.000


### Experiment 4B — Inspect top-weighted features

In [12]:
feature_names = np.array(tfidf.get_feature_names_out())

for i in range(X_tfidf.shape[0]):
    row = X_tfidf.getrow(i).toarray().ravel()
    top = row.argsort()[::-1][:5]
    print(f"D{i}: {corpus[i]}")
    print(list(zip(feature_names[top], row[top].round(3))))
    print()

D0: BERT-base achieved 91.3 F1 on CoNLL-2003.
[('bert', np.float64(0.377)), ('base', np.float64(0.377)), ('conll', np.float64(0.377)), ('f1', np.float64(0.377)), ('achieved', np.float64(0.377))]

D1: RoBERTa-large reached 94.8 accuracy on SST-2.
[('roberta', np.float64(0.407)), ('sst', np.float64(0.407)), ('large', np.float64(0.407)), ('reached', np.float64(0.407)), ('94', np.float64(0.407))]

D2: T5-small obtained 27.6 BLEU on WMT14 En-De.
[('wmt14', np.float64(0.346)), ('t5', np.float64(0.346)), ('small', np.float64(0.346)), ('obtained', np.float64(0.346)), ('en', np.float64(0.346))]

D3: DistilBERT scored 90% accuracy on AG News.
[('scored', np.float64(0.407)), ('90', np.float64(0.407)), ('distilbert', np.float64(0.407)), ('news', np.float64(0.407)), ('ag', np.float64(0.407))]

D4: The dataset contains 12,000 sentences and was released in 2003.
[('was', np.float64(0.311)), ('the', np.float64(0.311)), ('sentences', np.float64(0.311)), ('released', np.float64(0.311)), ('contains', np.

### Experiment 4C — Compare counts and TF-IDF for a repeated common word

In [13]:
weight_docs = [
    "model model model reports accuracy",
    "model reports f1",
    "model reports bleu",
    "dataset reports statistics",
]

cv = CountVectorizer()
tv = TfidfVectorizer()
C = cv.fit_transform(weight_docs)
T = tv.fit_transform(weight_docs)

print("Count vocabulary:", cv.get_feature_names_out())
print(pd.DataFrame(C.toarray(), columns=cv.get_feature_names_out()))

print("\nTF-IDF vocabulary:", tv.get_feature_names_out())
print(pd.DataFrame(T.toarray(), columns=tv.get_feature_names_out()).round(3))

Count vocabulary: ['accuracy' 'bleu' 'dataset' 'f1' 'model' 'reports' 'statistics']
   accuracy  bleu  dataset  f1  model  reports  statistics
0         1     0        0   0      3        1           0
1         0     0        0   1      1        1           0
2         0     1        0   0      1        1           0
3         0     0        1   0      0        1           1

TF-IDF vocabulary: ['accuracy' 'bleu' 'dataset' 'f1' 'model' 'reports' 'statistics']
   accuracy   bleu  dataset     f1  model  reports  statistics
0      0.45  0.000    0.000  0.000  0.862    0.235       0.000
1      0.00  0.000    0.000  0.772  0.492    0.403       0.000
2      0.00  0.772    0.000  0.000  0.492    0.403       0.000
3      0.00  0.000    0.663  0.000  0.000    0.346       0.663


**What we observe:** in Experiment 4C, `reports` appears once in every one of the four documents, so
it never becomes a document's most important word under TF-IDF even though it has the same raw count
(1) as several other, more discriminative terms — while `model` (repeated three times in document 0)
does dominate document 0's TF-IDF row, since within-document repetition still increases raw term
frequency even after the cross-document idf discount is applied. This is the core TF-IDF idea: a term's
weight blends "how often it appears here" with "how rare it is across the whole collection."

### Small self-check — Part 4

14. **A word that receives relatively low TF-IDF because it occurs in many documents. Show the evidence.**

In [14]:
doc_freq_on = int((X_count[:, list(count_vec.get_feature_names_out()).index("on")].toarray() > 0).sum())
print("'on' appears in", doc_freq_on, "of", len(corpus), "documents")
print("idf('on') =", round(tfidf.idf_[list(tfidf.get_feature_names_out()).index("on")], 3))
print("\nTF-IDF value of 'on' in every document:")
print(TF["on"])

'on' appears in 4 of 6 documents
idf('on') = 1.336

TF-IDF value of 'on' in every document:
D0    0.223841
D1    0.241706
D2    0.205282
D3    0.241706
D4    0.000000
D5    0.000000
Name: on, dtype: float64


`on` appears in 4 of the 6 documents (its raw count of 4 was even the single highest word-count in
Experiment 3B), giving it a low idf of about 1.34 — noticeably lower than domain-specific terms such as
`accuracy` (idf ≈ 1.85) or `bert` (idf ≈ 2.25). Consistent with that, `on` never shows up in any document's
top-5 TF-IDF terms in Experiment 4B: being common across the collection keeps its weight modest even
in the one document where it appears twice.

15. **A high-weight term from two different documents, and why it is discriminative here.**

In [15]:
print(TF["accuracy"])

D0    0.000000
D1    0.334091
D2    0.000000
D3    0.334091
D4    0.000000
D5    0.000000
Name: accuracy, dtype: float64


`accuracy` gets an identical TF-IDF weight of about 0.334 in both D1 (`RoBERTa-large ... accuracy on
SST-2`) and D3 (`DistilBERT ... accuracy on AG News`) — the two documents that share it. It is
discriminative precisely because it appears in exactly these two documents and no others: it cleanly
separates the "accuracy-metric" sentences from the F1-metric, BLEU-metric, and non-metric sentences in
the rest of the corpus.

16. **Does TF-IDF solve synonymy? Test "high accuracy" vs. "strong performance".**

In [16]:
syn_docs = ["high accuracy", "strong performance"]
syn_tfidf = TfidfVectorizer()
S = syn_tfidf.fit_transform(syn_docs)

print("vocabulary:", syn_tfidf.get_feature_names_out())
print("cosine similarity:", cosine_similarity(S)[0, 1])

vocabulary: ['accuracy' 'high' 'performance' 'strong']
cosine similarity: 0.0


No. The cosine similarity between "high accuracy" and "strong performance" is exactly **0.0**,
because the two phrases share zero vocabulary words — TF-IDF only re-weights the terms a document
already contains, it never notices that "accuracy" and "performance" (or "high" and "strong") are related
concepts. Synonymy is a distributional/semantic relationship that a purely lexical, count-based
representation has no mechanism to capture.

---
## Part 5 — From Vectors to Similarity and Retrieval

### Experiment 5A — Corpus similarity matrix

In [17]:
similarity = cosine_similarity(X_tfidf)
sim_df = pd.DataFrame(
    similarity,
    index=[f"D{i}" for i in range(len(corpus))],
    columns=[f"D{i}" for i in range(len(corpus))],
)
sim_df.round(3)

,D0,D1,D2,D3,D4,D5
D0,1.000,0.054,0.046,0.054,0.079,0.000
D1,0.054,1.000,0.050,0.170,0.000,0.000
D2,0.046,0.050,1.000,0.050,0.000,0.000
D3,0.054,0.170,0.050,1.000,0.000,0.000
D4,0.079,0.000,0.000,0.000,1.000,0.075
D5,0.000,0.000,0.000,0.000,0.075,1.000


### Experiment 5B — Build a tiny TF-IDF retriever

In [18]:
query = "classification model reports strong accuracy"
query_vec = tfidf.transform([query])
scores = cosine_similarity(query_vec, X_tfidf).ravel()
ranking = np.argsort(scores)[::-1]

for rank, idx in enumerate(ranking[:3], start=1):
    print(rank, round(float(scores[idx]), 3), corpus[idx])

1 0.334 RoBERTa-large reached 94.8 accuracy on SST-2.
2 0.334 DistilBERT scored 90% accuracy on AG News.
3 0.0 The dataset contains 12,000 sentences and was released in 2003.


### Experiment 5C — Compare cosine with token-set Jaccard

In [19]:
def token_set(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def jaccard(a, b):
    A, B = token_set(a), token_set(b)
    return len(A & B) / len(A | B) if (A | B) else 0.0

query = "classification model reports strong accuracy"
for i, text in enumerate(corpus):
    print(
        f"D{i}",
        "Jaccard=", round(jaccard(query, text), 3),
        "Cosine=", round(float(scores[i]), 3),
    )

D0 Jaccard= 0.0 Cosine= 0.0
D1 Jaccard= 0.077 Cosine= 0.334
D2 Jaccard= 0.0 Cosine= 0.0
D3 Jaccard= 0.091 Cosine= 0.334
D4 Jaccard= 0.0 Cosine= 0.0
D5 Jaccard= 0.0 Cosine= 0.0


**What we observe:** the query never appears verbatim anywhere in the corpus, yet cosine similarity
still produces a graded, usable ranking — D1 and D3 (the two accuracy-metric sentences) tie for the top
score of 0.334, while the F1/BLEU/dataset/training sentences all score much lower or zero. Jaccard uses
the same idea of set overlap but ignores term weighting entirely, so it ranks purely by how large the
shared-word set is relative to the union — here it happens to agree on which documents are non-zero, but
disagrees on the finer ordering (e.g. D3's Jaccard of 0.091 edges out D1's 0.077, while cosine ties them),
because Jaccard cannot tell that "accuracy" is a rarer, more diagnostic overlap than the incidental
overlap on function words.

### Small self-check — Part 5

17. **Change the query to "translation score on WMT" and report the top three documents.**

In [20]:
query2 = "translation score on WMT"
query_vec2 = tfidf.transform([query2])
scores2 = cosine_similarity(query_vec2, X_tfidf).ravel()
ranking2 = np.argsort(scores2)[::-1]

for rank, idx in enumerate(ranking2[:3], start=1):
    print(rank, round(float(scores2[idx]), 3), corpus[idx])

1 0.242 RoBERTa-large reached 94.8 accuracy on SST-2.
2 0.242 DistilBERT scored 90% accuracy on AG News.
3 0.224 BERT-base achieved 91.3 F1 on CoNLL-2003.


None of the top-ranked documents actually discusses translation — they are pulled up purely because
they share the common word `on` with the query, which is a good early warning of the lexical-mismatch
failure mode explored properly in Part 6.

18. **Construct one query for which cosine and Jaccard produce different rankings. Explain why.** The query `"classification model reports strong accuracy"` above already does this: cosine ties D1 and D3 at 0.334, while Jaccard ranks D3 (0.091) ahead of D1 (0.077). Jaccard only counts *how many* distinct words overlap relative to the union size, so it is sensitive to each sentence's overall length; cosine additionally weights each overlapping word by its idf, so it can end up ranking documents differently once term rarity is taken into account, even when the raw overlapping-word sets look similar.
19. **Does a score of 0.8 mean "80% semantically equivalent"? Explain why not.** No. Cosine similarity is a geometric angle between two sparse, lexical-count vectors — it reflects how much vocabulary (weighted by rarity) two texts share, not how similar their meanings are. Two paraphrases with zero shared words score 0.0 despite being fully equivalent in meaning, and two unrelated sentences that happen to share several common function words can score well above 0. The number is a property of the representation and metric, not a calibrated percentage of "meaning" preserved.

---
## Part 6 — Representation Stress Tests: What Does the Vector Forget?

### Experiment 6A — Word-order failure

In [21]:
order_probe = [
    "model beats baseline",
    "baseline beats model",
]

u = CountVectorizer(ngram_range=(1, 1))
U = u.fit_transform(order_probe)
print("Unigram features:", u.get_feature_names_out())
print(U.toarray())
print("unigram cosine:", cosine_similarity(U)[0, 1])

b = CountVectorizer(ngram_range=(1, 2))
B = b.fit_transform(order_probe)
print("\nUnigram + bigram features:", b.get_feature_names_out())
print("unigram+bigram cosine:", cosine_similarity(B)[0, 1])

Unigram features: ['baseline' 'beats' 'model']
[[1 1 1]
 [1 1 1]]
unigram cosine: 1.0000000000000002

Unigram + bigram features: ['baseline' 'baseline beats' 'beats' 'beats baseline' 'beats model'
 'model' 'model beats']
unigram+bigram cosine: 0.6


### Experiment 6B — Lexical mismatch / paraphrase failure

In [22]:
paraphrases = [
    "the model achieved high accuracy",
    "the system obtained strong performance",
    "the dataset contains twelve thousand examples",
]

pv = TfidfVectorizer()
P = pv.fit_transform(paraphrases)
pd.DataFrame(cosine_similarity(P)).round(3)

,0,1,2
0,1.000,0.080,0.072
1,0.080,1.000,0.072
2,0.072,0.072,1.000


### Experiment 6C — Repetition sensitivity

In [23]:
repeat_probe = [
    "model reports accuracy",
    "model model model model reports accuracy",
]

cv = CountVectorizer()
tv = TfidfVectorizer()
C = cv.fit_transform(repeat_probe)
T = tv.fit_transform(repeat_probe)

print("Count vectors:")
print(C.toarray())
print("Count cosine:", cosine_similarity(C)[0, 1])

print("\nTF-IDF vectors:")
print(T.toarray().round(3))
print("TF-IDF cosine:", cosine_similarity(T)[0, 1])

Count vectors:
[[1 1 1]
 [1 4 1]]
Count cosine: 0.8164965809277261

TF-IDF vectors:
[[0.577 0.577 0.577]
 [0.236 0.943 0.236]]
TF-IDF cosine: 0.8164965809277261


**Representation diagnosis table**

| Representation | What it preserves reasonably well | What remains unresolved |
|---|---|---|
| Unigram BoW | Vocabulary counts | Most word order, syntax, long-range relations |
| n-gram counts | Some local order | Rapidly growing dimensionality; sparse unseen phrases |
| TF-IDF | Lexical distinctiveness | Context, synonymy, polysemy, relations |
| Subword tokenization | Coverage of rare/new surface forms | Does not by itself create semantic similarity |

**What we observe:**
- **6A:** unigram counts give `model beats baseline` and `baseline beats model` a cosine similarity of
  **1.0** — a perfect match despite opposite meaning. Adding bigrams drops the score to **0.6**, since
  bigrams like `beats baseline` vs. `beats model` are now different dimensions, but the score is still far
  from 0 because both sentences still share the bigram `beats`-adjacent structure and every unigram.
- **6B:** all three paraphrase-style sentences score low (0.07–0.08) against each other even though a
  human reader would call the first two closely related in topic — none of them share enough literal
  vocabulary for TF-IDF cosine to notice.
- **6C:** repeating "model" four extra times barely moves the cosine similarity (0.816 for both raw counts
  and TF-IDF) in this particular two-document, fully-overlapping-vocabulary case — cosine's own
  normalization step absorbs much of the repetition's effect on direction, and here every word happens to
  appear in both documents so idf weighting cancels out identically for the two vectors, leaving TF-IDF and
  raw-count cosine numerically identical.

### Small self-check — Part 6

20. **Why are "model beats baseline" and "baseline beats model" a strong counterexample for unigram BoW?** They contain identical words with identical counts, so their unigram vectors are literally the same point in vector space (cosine = 1.0) — yet the two sentences report opposite outcomes. This shows unigram BoW cannot represent grammatical role or word order at all, only vocabulary membership and frequency.
21. **Why might bigrams help this pair but still fail on long-distance relationships?** Bigrams capture *adjacent* word order, so `beats baseline` and `beats model` become distinguishable dimensions, lowering the similarity from 1.0 to 0.6. But a bigram window only ever spans two neighbouring tokens, so any relationship that depends on words further apart in the sentence (e.g. a subject and a verb separated by a long clause) still falls outside what n-grams of size 2 can represent — you would need increasingly large, increasingly sparse n-grams to reach further, which was already flagged as a cost of n-grams in Part 3.
22. **A synonym-based paraphrase pair receiving lower similarity than expected.** `"high accuracy"` vs. `"strong performance"` (Part 4, self-check 16) — cosine similarity of exactly 0.0 despite the two phrases meaning almost the same thing to a human reader, because they share no vocabulary at all.
23. **One distinction TF-IDF cannot represent even with a very large corpus.** TF-IDF cannot represent polysemy/word-sense — a word like "bank" gets exactly one dimension and one weight per document regardless of whether it means a riverbank or a financial institution in that sentence; no amount of additional documents changes the fact that TF-IDF assigns a single, context-free weight per surface word.

---
## Exit Ticket

24. **One thing a sparse vector preserves:** which vocabulary words a document contains, and roughly how often each one occurs.
25. **One thing a sparse vector forgets:** the order in which words occur (e.g. "model beats baseline" vs. "baseline beats model" collapse to the same unigram vector).
26. **One reason subword tokenization is useful:** it gives near-complete vocabulary coverage — any rare, novel, or misspelled word can still be represented as a sequence of known smaller pieces instead of becoming an unrepresentable out-of-vocabulary token.
27. **One reason TF-IDF is still not a semantic representation:** it assigns weights purely from surface word identity and corpus-wide frequency, so two synonymous phrases with no shared vocabulary (e.g. "high accuracy" vs. "strong performance") get a similarity of 0.
28. **One question a learned embedding should answer better:** whether two sentences that describe the same idea in different words (e.g. "strong performance" and "high accuracy") are actually close in meaning, even with zero lexical overlap.

---
# Part B — Final Takeaway Homework: Representation Stress Test and Sparse Retriever

In [24]:
home_corpus = [
    "BERT-base achieved 91.3 F1 on CoNLL-2003.",
    "RoBERTa-large reached 94.8 accuracy on SST-2.",
    "T5-small obtained 27.6 BLEU on WMT14 En-De.",
    "DistilBERT scored 90% accuracy on AG News.",
    "BART-large reported ROUGE-L 41.2 on CNN/DailyMail.",
    "DeBERTa-v3-large reached 90.0 accuracy on BoolQ.",
    "ELECTRA-large reported 90.9 accuracy on MNLI-m.",
    "The corpus contains 91.3 million tokens.",
    "Training used 16 epochs and batch size 32.",
    "The learning rate was 2e-5.",
    "The model outperformed the baseline on classification.",
    "The baseline outperformed the model on classification.",
    "The system obtained strong performance on a sentiment benchmark.",
    "A translation system produced a competitive BLEU score.",
    "The dataset was released in 2024 and contains 12,000 examples.",
    "No numerical evaluation result was reported for GPT-2.",
]
len(home_corpus)

16

## Task A — Tokenization audit

Eight challenge strings, one per required category, are probed on all three pretrained tokenizers.

In [25]:
challenge_strings = [
    "representation",                     # ordinary English word
    "hyperparameterization",              # rare / technical compound
    "DeBERTa-v3-large",                   # model name with punctuation
    "WMT14-En-De 27.6",                   # number-containing string
    "neural.nlp@dau.ac.in",               # email-like string
    "café naïve résumé",                  # multilingual / accented string
    "accuarcy",                           # misspelling of "accuracy"
    "IT594#Assignment2!!!",               # challenge item of my choice
]

tokenization_rows = []
for text in challenge_strings:
    for name, tok in tokenizers.items():
        pieces = tok.tokenize(text)
        tokenization_rows.append({
            "text": text,
            "tokenizer": name,
            "tokens": " | ".join(pieces),
            "n_tokens": len(pieces),
        })

tokenization_audit = pd.DataFrame(tokenization_rows)
tokenization_audit.to_csv("tokenization_audit.csv", index=False)
tokenization_audit

,text,tokenizer,tokens,n_tokens
0,representation,WordPiece / BERT,representation,1
1,representation,BPE / GPT-2,represent | ation,2
2,representation,Unigram / T5,▁representation,1
3,hyperparameterization,WordPiece / BERT,hyper | ##para | ##meter | ##ization,4
4,hyperparameterization,BPE / GPT-2,hyper | param | eter | ization,4
5,hyperparameterization,Unigram / T5,▁hyper | para | meter | ization,4
6,DeBERTa-v3-large,WordPiece / BERT,de | ##bert | ##a | - | v | ##3 | - | large,8
7,DeBERTa-v3-large,BPE / GPT-2,De | BER | Ta | - | v | 3 | - | large,8
8,DeBERTa-v3-large,Unigram / T5,▁De | BER | T | a | - | v | 3 | - | large,9
9,WMT14-En-De 27.6,WordPiece / BERT,w | ##mt | ##14 | - | en | - | de | 27 | . | 6,10


**Observation to fill in from your own run:** for each row above, add a one-sentence note — e.g.
whether the string stayed close to whole-word length, fragmented heavily, or was handled inconsistently
across the three tokenizer families. In general, expect the ordinary word to need the fewest tokens across
all three tokenizers, the rare compound and the accented phrase to fragment the most, and the email-like
and punctuation-heavy strings to be split at almost every punctuation mark since none of these tokenizers
treat `@`, `.`, or `#` as meaningful separators.

## Task B — Build three sparse representations

In [26]:
representations = {
    "count_unigram": CountVectorizer(ngram_range=(1, 1)),
    "count_uni_bigram": CountVectorizer(ngram_range=(1, 2)),
    "tfidf_unigram": TfidfVectorizer(ngram_range=(1, 1)),
}

summary = []
fitted_representations = {}
for name, vectorizer in representations.items():
    X = vectorizer.fit_transform(home_corpus)
    fitted_representations[name] = (vectorizer, X)
    density = X.nnz / (X.shape[0] * X.shape[1])
    summary.append({
        "representation": name,
        "n_documents": X.shape[0],
        "n_features": X.shape[1],
        "nnz": X.nnz,
        "density": round(density, 4),
    })

representation_summary = pd.DataFrame(summary)
representation_summary

,representation,n_documents,n_features,nnz,density
0,count_unigram,16,81,118,0.0910
1,count_uni_bigram,16,174,222,0.0797
2,tfidf_unigram,16,81,118,0.0910


Moving from unigrams to unigrams+bigrams more than doubles the feature count (81 → 174) while the
matrix stays about as sparse overall — bigrams add many columns that are each non-zero in only a single
document.

## Task C — Build a sparse retriever

Using TF-IDF (unigrams + bigrams) and cosine similarity.

In [27]:
tfidf_home = TfidfVectorizer(ngram_range=(1, 2))
X_home = tfidf_home.fit_transform(home_corpus)

def retrieve(query, k=3):
    q = tfidf_home.transform([query])
    scores = cosine_similarity(q, X_home).ravel()
    order = np.argsort(scores)[::-1][:k]
    return [
        {"rank": r + 1, "index": int(i), "score": round(float(scores[i]), 3), "text": home_corpus[i]}
        for r, i in enumerate(order)
    ]

queries = [
    "classification model with high accuracy",
    "translation result measured by BLEU",
    "dataset size and release information",
    "model beats baseline",
    "strong sentiment benchmark performance",
]

retrieval_rows = []
for query in queries:
    print("\nQUERY:", query)
    for item in retrieve(query):
        print(item)
        row = dict(item)
        row["query"] = query
        retrieval_rows.append(row)

retrieval_results = pd.DataFrame(retrieval_rows)
retrieval_results.to_csv("retrieval_results.csv", index=False)


QUERY: classification model with high accuracy
{'rank': 1, 'index': 10, 'score': 0.342, 'text': 'The model outperformed the baseline on classification.'}
{'rank': 2, 'index': 11, 'score': 0.342, 'text': 'The baseline outperformed the model on classification.'}
{'rank': 3, 'index': 6, 'score': 0.116, 'text': 'ELECTRA-large reported 90.9 accuracy on MNLI-m.'}

QUERY: translation result measured by BLEU
{'rank': 1, 'index': 13, 'score': 0.326, 'text': 'A translation system produced a competitive BLEU score.'}
{'rank': 2, 'index': 15, 'score': 0.16, 'text': 'No numerical evaluation result was reported for GPT-2.'}
{'rank': 3, 'index': 2, 'score': 0.115, 'text': 'T5-small obtained 27.6 BLEU on WMT14 En-De.'}

QUERY: dataset size and release information
{'rank': 1, 'index': 8, 'score': 0.276, 'text': 'Training used 16 epochs and batch size 32.'}
{'rank': 2, 'index': 14, 'score': 0.24, 'text': 'The dataset was released in 2024 and contains 12,000 examples.'}
{'rank': 3, 'index': 13, 'score':

## Task D — Compare cosine and Jaccard ranking for all five queries

In [28]:
def token_set(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def jaccard(a, b):
    A, B = token_set(a), token_set(b)
    return len(A & B) / len(A | B) if (A | B) else 0.0

for query in queries:
    q_vec = tfidf_home.transform([query])
    cos_scores = cosine_similarity(q_vec, X_home).ravel()
    cos_rank = np.argsort(cos_scores)[::-1][:3]

    jac_scores = np.array([jaccard(query, text) for text in home_corpus])
    jac_rank = np.argsort(jac_scores)[::-1][:3]

    print("\nQUERY:", query)
    print("cosine top-3: ", [(int(i), round(float(cos_scores[i]), 3)) for i in cos_rank])
    print("jaccard top-3:", [(int(i), round(float(jac_scores[i]), 3)) for i in jac_rank])


QUERY: classification model with high accuracy
cosine top-3:  [(10, 0.342), (11, 0.342), (6, 0.116)]
jaccard top-3: [(10, 0.222), (11, 0.222), (3, 0.091)]

QUERY: translation result measured by BLEU
cosine top-3:  [(13, 0.326), (15, 0.16), (2, 0.115)]
jaccard top-3: [(13, 0.2), (15, 0.077), (2, 0.071)]

QUERY: dataset size and release information
cosine top-3:  [(8, 0.276), (14, 0.24), (13, 0.0)]
jaccard top-3: [(8, 0.182), (14, 0.143), (13, 0.0)]

QUERY: model beats baseline
cosine top-3:  [(10, 0.395), (11, 0.395), (15, 0.0)]
jaccard top-3: [(10, 0.286), (11, 0.286), (15, 0.0)]

QUERY: strong sentiment benchmark performance
cosine top-3:  [(12, 0.618), (15, 0.0), (14, 0.0)]
jaccard top-3: [(12, 0.444), (15, 0.0), (14, 0.0)]


**Case 1 — query `"classification model with high accuracy"`.** Cosine's rank-3 pick is document 6
(`ELECTRA-large reported 90.9 accuracy on MNLI-m.`), while Jaccard's rank-3 pick is document 3
(`DistilBERT scored 90% accuracy on AG News.`). Both share the word `accuracy` with the query, but idf
weighting inside TF-IDF prefers a slightly different one of the two once the rarer bigram overlaps are
taken into account, whereas Jaccard only compares raw set sizes and picks the other one.

**Case 2 — an additional constructed query, `"the accuracy of the model on the classification task"`,
makes the divergence sharper:**

In [29]:
extra_query = "the accuracy of the model on the classification task"

q_vec = tfidf_home.transform([extra_query])
cos_scores = cosine_similarity(q_vec, X_home).ravel()
cos_rank = np.argsort(cos_scores)[::-1][:3]

jac_scores = np.array([jaccard(extra_query, text) for text in home_corpus])
jac_rank = np.argsort(jac_scores)[::-1][:3]

print("cosine top-3: ", [(int(i), round(float(cos_scores[i]), 3), home_corpus[i]) for i in cos_rank])
print("jaccard top-3:", [(int(i), round(float(jac_scores[i]), 3), home_corpus[i]) for i in jac_rank])

cosine top-3:  [(11, 0.674, 'The baseline outperformed the model on classification.'), (10, 0.555, 'The model outperformed the baseline on classification.'), (9, 0.143, 'The learning rate was 2e-5.')]
jaccard top-3: [(10, 0.444, 'The model outperformed the baseline on classification.'), (11, 0.444, 'The baseline outperformed the model on classification.'), (3, 0.167, 'DistilBERT scored 90% accuracy on AG News.')]


Document 10 (`The model outperformed the baseline on classification.`) and document 11
(`The baseline outperformed the model on classification.`) are mirror-image sentences that use exactly
the same words. Jaccard, being a pure set-overlap measure, correctly scores them identically (0.444
each). Cosine similarity with bigrams, however, ranks document 11 clearly above document 10 (0.674 vs.
0.555) — a byproduct of which two-word windows happen to align with the query's own bigrams, not a real
difference in relevance. This is a case where the "smarter," weighted metric introduces an artefact that
the simpler, unweighted metric does not.

## Task E — Required representation stress tests

In [30]:
# 1. Word order
pair_order = [
    "the model outperformed the baseline on classification",
    "the baseline outperformed the model on classification",
]
u = CountVectorizer()
U = u.fit_transform(pair_order)
b = CountVectorizer(ngram_range=(1, 2))
B = b.fit_transform(pair_order)
print("Word order  | unigram cosine:", round(cosine_similarity(U)[0, 1], 3),
      " uni+bigram cosine:", round(cosine_similarity(B)[0, 1], 3))

Word order  | unigram cosine: 1.0  uni+bigram cosine: 0.867


*Diagnosis:* unigram cosine is a perfect **1.0** — swapping "model" and "baseline" leaves the
unigram vector completely unchanged, so the representation cannot say who beat whom. Adding bigrams
lowers it to **0.867**, since bigrams like "model outperformed" vs. "baseline outperformed" now differ —
but the score is still very high, because most of the sentence's bigrams (e.g. "on classification",
"outperformed the") are shared regardless of the swap.

In [31]:
# 2. Synonym / paraphrase
pair_synonym = ["high accuracy", "strong performance"]
tv = TfidfVectorizer()
T = tv.fit_transform(pair_synonym)
print("Synonym pair cosine:", round(cosine_similarity(T)[0, 1], 3))

Synonym pair cosine: 0.0


*Diagnosis:* cosine similarity is **0.0**. Lexical mismatch is total here — TF-IDF has no way to know
that "high"/"strong" or "accuracy"/"performance" are related, since it only ever compares surface tokens.

In [32]:
# 3. Repetition
pair_repeat = [
    "model reports accuracy",
    "model model model model model reports accuracy",
]
cv = CountVectorizer()
C = cv.fit_transform(pair_repeat)
tv2 = TfidfVectorizer()
T2 = tv2.fit_transform(pair_repeat)
print("Repetition  | count cosine:", round(cosine_similarity(C)[0, 1], 3),
      " tfidf cosine:", round(cosine_similarity(T2)[0, 1], 3))

Repetition  | count cosine: 0.778  tfidf cosine: 0.778


*Diagnosis:* repeating "model" five times barely moves either score (both land at **0.778**), and
count cosine and TF-IDF cosine come out identical. With only two documents that share every distinct
word, idf weighting is the same constant for every term in both vectors, so TF-IDF direction is just a
scaled copy of the raw-count direction — cosine similarity, which is scale-invariant, cannot tell the two
representations apart in this particular case.

In [33]:
# 4. Novel surface form
pair_novel = [
    "The model achieved strong accuracy.",
    "The Zephyr-X9 achieved strong accuracy.",
]
tv3 = TfidfVectorizer()
T3 = tv3.fit_transform(pair_novel)
print("Novel-form vocabulary:", tv3.get_feature_names_out())
print("Novel-form cosine:", round(cosine_similarity(T3)[0, 1], 3))

print("\nHow subword tokenizers react to the unseen model name:")
for name, tok in tokenizers.items():
    pieces = tok.tokenize("Zephyr-X9")
    print(f"{name:18s}", pieces)

Novel-form vocabulary: ['accuracy' 'achieved' 'model' 'strong' 'the' 'x9' 'zephyr']
Novel-form cosine: 0.58

How subword tokenizers react to the unseen model name:
WordPiece / BERT   ['ze', '##phy', '##r', '-', 'x', '##9']
BPE / GPT-2        ['Z', 'eph', 'yr', '-', 'X', '9']
Unigram / T5       ['▁Ze', 'phy', 'r', '-', 'X', '9']


*Diagnosis:* at the word level, `"model"` and `"Zephyr-X9"` are simply two different vocabulary
items, so TF-IDF cosine drops to about 0.58 even though the surrounding sentence is identical — the
representation has no notion that both are plausibly "a model name" in this context. Subword
tokenization softens the raw out-of-vocabulary problem: instead of one unrepresentable token, each
tokenizer above breaks the invented name into smaller known pieces so it can still be encoded at all — but
having a token *sequence* for "Zephyr-X9" does not by itself tell any downstream sparse representation
that this sequence plays the same syntactic role as "model."

*5. Local order* is already covered by the word-order test above: moving from unigrams to
unigrams+bigrams recovers exactly the "who is adjacent to whom" information that the word-order stress
test needs, without recovering anything about relationships beyond a two-word window.

## Task F — Error analysis and bridge forward

Five surprising outcomes drawn directly from the retrieval and stress-test results computed above.

| # | Example / query | Expected | Observed | Cause | Possible next representation |
|---|---|---|---|---|---|
| 1 | `"dataset size and release information"` | Doc 14 ("released in 2024 and contains 12,000 examples") ranks first | Doc 8 ("Training used 16 epochs and batch size 32.") ranks first (0.276 vs. 0.240) | Doc 8 shares the word "size" and several common connective words/bigrams with the query, while the topical match (doc 14) shares fewer exact surface tokens | Distributional word vectors, so "dataset release" and "epoch/batch size" are not pulled together just because both contain "size" |
| 2 | `"classification model with high accuracy"` | The system should indicate which of the two mirror sentences (doc 10 "model outperformed baseline" vs. doc 11 "baseline outperformed model") is the better match | Both tie exactly at 0.342 | Query bigrams ("classification model") don't align with either sentence's bigrams in a way that favours one direction over the other | A representation sensitive to grammatical subject/object role, not just adjacency |
| 3 | `"translation result measured by BLEU"` | Doc 2, the only sentence with an explicit BLEU score, should rank highest | Doc 13 ranks first (correct), but doc 15 ("No numerical evaluation result was reported for GPT-2.") outranks doc 2 for rank 2/3 (0.16 vs. 0.115) | "result" is rarer corpus-wide than "BLEU" (which appears in 2 docs vs. "result" appearing in only 1), so idf briefly favours the accidental overlap over the on-topic term | A representation that weighs semantic relevance to "BLEU/translation" rather than raw corpus rarity |
| 4 | `"model beats baseline"` | Three genuinely related documents in top-3 | Rank 3 is doc 15 (`"No numerical evaluation result was reported for GPT-2."`) at a similarity score of exactly 0.0 | Only two documents in the corpus are lexically related to "model"/"baseline"; forcing `k=3` pads the list with an unrelated, zero-similarity document | Retrieval should threshold on a minimum score rather than always returning exactly `k` results |
| 5 | Constructed query `"the accuracy of the model on the classification task"` | Doc 10 and doc 11 (mirror sentences) should score identically, since Jaccard already treats them as identical | Cosine ranks doc 11 well above doc 10 (0.674 vs. 0.555) | Bigram-boundary alignment differs slightly between the two mirror sentences relative to the query's own bigrams, purely as an artefact of where each word happens to sit | A representation that captures meaning/role rather than exact adjacency windows |



## Written takeaway (300–400 words)

**29. Which distinctions were preserved well by counts or TF-IDF?**
Both representations reliably separated documents by topic vocabulary: the four accuracy-benchmark
sentences (RoBERTa, DistilBERT, DeBERTa, ELECTRA) consistently scored higher against accuracy-flavoured
queries than the F1, BLEU, or training-configuration sentences did, and TF-IDF correctly pushed corpus-wide
common words like "on" and "the" down in importance (Part 4, self-check 14) while keeping genuinely
discriminative words like "accuracy" (self-check 15) and "BLEU" weighted highly. Retrieval built purely on
TF-IDF and cosine similarity (Task C) produced sensible top-1 results for every one of the five homework
queries.

**30. Which distinctions were lost even when cosine similarity was used?**
Cosine similarity is still built on top of a lexical, order-blind vector, so it inherited every one of that
vector's blind spots. Word order was preserved only as far as the n-gram window: unigram cosine gave
"model beats baseline" and "baseline beats model" a perfect similarity of 1.0 (Part 6, Experiment 6A), and
even the bigram-augmented "model outperformed baseline" pair in Task E still scored a high 0.867 despite
describing opposite outcomes. Synonymy was not recovered at all — "high accuracy" vs. "strong
performance" scored exactly 0.0 in both Part 4 and Task E. And idf-driven weighting could actively
mislead retrieval, as in Task F's error #3, where a corpus-wide rarity accident let an unrelated sentence
outrank the one document that actually contained the query's key metric term, "BLEU."

**31. What did adding bigrams improve, and what did it cost?**
Bigrams recovered some local word-order information — enough to drop the word-order stress test's
cosine similarity from a perfect 1.0 down to 0.6–0.867 depending on the sentence pair — and let the Task C
retriever match short local phrases like "model beats baseline" more precisely. The cost was a large jump
in feature count (81 → 174 features on the same 16-document homework corpus, Task B) with most new
bigram columns firing in only a single document, so the representation grew sparser and higher-dimensional
without extending word-order sensitivity beyond a two-token window.

**32. How did subword tokenization change coverage without automatically solving semantic similarity?**
Every probe string, no matter how rare or invented, could still be tokenized into some sequence of known
sub-word pieces by all three tokenizer families (Part 2, Task A) — there was no out-of-vocabulary failure.
But having a token sequence for an unseen model name like "Zephyr-X9" (Task E) did not stop the downstream
sparse vector from treating it as a completely unrelated dimension to "model," so tokenization solved
*coverage*, not *meaning*.

**33. What specific failure now motivates distributional or learned dense representations?**
The synonymy failure is the sharpest motivator: two phrases a person would call nearly identical in
meaning, "high accuracy" and "strong performance," scored a cosine similarity of exactly 0.0 in every
sparse representation tested here, because none of these methods place related words near each other in
vector space — only a learned, distributional representation can.

## Submission structure

```
assignment_02/
├── representation_lab.ipynb   <- this notebook
├── tokenization_audit.csv     <- written by Task A
├── retrieval_results.csv      <- written by Task C
├── stress_tests.csv           <- written by the cell below
└── README.md
```

In [34]:
stress_test_rows = [
    {
        "stress_test": "Word order",
        "controlled_change": "model outperformed baseline vs. baseline outperformed model",
        "unigram_cosine": round(cosine_similarity(u.transform(pair_order))[0, 1], 3),
        "uni_bigram_cosine": round(cosine_similarity(b.transform(pair_order))[0, 1], 3),
    },
    {
        "stress_test": "Synonym / paraphrase",
        "controlled_change": "high accuracy vs. strong performance",
        "cosine": round(cosine_similarity(tv.transform(pair_synonym))[0, 1], 3),
    },
    {
        "stress_test": "Repetition",
        "controlled_change": "model reports accuracy vs. 5x model reports accuracy",
        "count_cosine": round(cosine_similarity(cv.transform(pair_repeat))[0, 1], 3),
        "tfidf_cosine": round(cosine_similarity(tv2.transform(pair_repeat))[0, 1], 3),
    },
    {
        "stress_test": "Novel surface form",
        "controlled_change": "model vs. Zephyr-X9 (unseen model name)",
        "cosine": round(cosine_similarity(tv3.transform(pair_novel))[0, 1], 3),
    },
    {
        "stress_test": "Local order",
        "controlled_change": "unigram vs. unigram+bigram on the word-order pair",
        "unigram_cosine": round(cosine_similarity(u.transform(pair_order))[0, 1], 3),
        "uni_bigram_cosine": round(cosine_similarity(b.transform(pair_order))[0, 1], 3),
    },
]

stress_tests_df = pd.DataFrame(stress_test_rows)
stress_tests_df.to_csv("stress_tests.csv", index=False)
stress_tests_df

,stress_test,controlled_change,unigram_cosine,uni_bigram_cosine,cosine,count_cosine,tfidf_cosine
0,Word order,model outperformed baseline vs. baseline outpe...,1.0,0.867,NaN,NaN,NaN
1,Synonym / paraphrase,high accuracy vs. strong performance,NaN,NaN,0.00,NaN,NaN
2,Repetition,model reports accuracy vs. 5x model reports ac...,NaN,NaN,NaN,0.778,0.778
3,Novel surface form,model vs. Zephyr-X9 (unseen model name),NaN,NaN,0.58,NaN,NaN
4,Local order,unigram vs. unigram+bigram on the word-order pair,1.0,0.867,NaN,NaN,NaN


### README.md content

```
# Assignment 2 — Representation Stress Test and Sparse Retriever

Author: Tirth (IT594, DAU)

## Contents
- representation_lab.ipynb — full guided lab (Parts 1-6) and take-home tasks A-F, runs top to bottom
- tokenization_audit.csv — Task A output: 8 challenge strings x 3 tokenizer families
- retrieval_results.csv — Task C output: top-3 retrieved sentences for 5 queries
- stress_tests.csv — Task E output: controlled-pair similarity scores for all 5 required stress tests

## How to run
Open representation_lab.ipynb in Google Colab (or any environment with internet access, since the
tokenizer cells download bert-base-uncased, gpt2, and t5-small from the Hugging Face Hub on first use)
and run all cells top to bottom.

## Key findings
- Sparse lexical representations (count vectors, TF-IDF) preserve vocabulary overlap and, to a limited
  extent, local word order once n-grams are added.
- They do not preserve synonymy: "high accuracy" and "strong performance" score 0.0 cosine similarity.
- Subword tokenization solves vocabulary coverage for rare/unseen strings but does not by itself create
  any semantic relationship between an unseen token sequence and known ones.
- These failures (see Task F) motivate the next module's move to distributional and learned dense
  representations.
```